# Module 1: Parameter Playground

Experiment with LLM configuration parameters and compare outputs side-by-side.
This notebook demonstrates how temperature, top-p, and max_tokens shape generation.

## Configuration

Set your provider and model below. The rest of the notebook adapts automatically.

In [ ]:
import sys
sys.path.insert(0, '..')

from utils.llm_client import call_llm
import pandas as pd
import matplotlib.pyplot as plt

# === CONFIGURE THIS CELL ===
PROVIDER = "openai"        # "openai", "anthropic", or "ollama"
MODEL = "gpt-4o"           # or "claude-sonnet-4-20250514", "llama3"
# ===========================

print(f"Provider: {PROVIDER} | Model: {MODEL}")

---
## 1. Temperature Sweep

Temperature reshapes the probability distribution. Lower = more peaked (deterministic), higher = more flat (creative).

In [ ]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]
PROMPT = "Explain quantum computing in one sentence for a software engineer."

temp_results = []
for temp in TEMPERATURES:
    try:
        response = call_llm(PROMPT, provider=PROVIDER, model=MODEL, temperature=temp, max_tokens=200)
        temp_results.append({"temperature": temp, "response": response})
        print(f"\n=== Temperature {temp} ===")
        print(response[:300])
    except Exception as e:
        print(f"Error at temp {temp}: {e}")
        temp_results.append({"temperature": temp, "response": f"ERROR: {e}"})

print(f"\nCompleted {len(temp_results)} temperature tests.")

### Temperature Results Table

In [ ]:
temp_df = pd.DataFrame(temp_results)
temp_df["response_preview"] = temp_df["response"].str[:120] + "..."
temp_df[["temperature", "response_preview"]]

---
## 2. Top-p (Nucleus Sampling) Sweep

Top-p truncates the vocabulary to the smallest set of tokens whose cumulative probability exceeds the threshold.

In [ ]:
TOP_P_VALUES = [0.1, 0.5, 0.9, 1.0]
PROMPT_TOP_P = "Describe a sunset over the ocean in one sentence."

top_p_results = []
for top_p in TOP_P_VALUES:
    try:
        response = call_llm(
            PROMPT_TOP_P,
            provider=PROVIDER,
            model=MODEL,
            temperature=0.7,
            top_p=top_p,
            max_tokens=200,
        )
        top_p_results.append({"top_p": top_p, "response": response})
        print(f"\n=== Top-p {top_p} ===")
        print(response[:300])
    except Exception as e:
        print(f"Error at top_p {top_p}: {e}")
        top_p_results.append({"top_p": top_p, "response": f"ERROR: {e}"})

print(f"\nCompleted {len(top_p_results)} top-p tests.")

In [ ]:
top_p_df = pd.DataFrame(top_p_results)
top_p_df["response_preview"] = top_p_df["response"].str[:120] + "..."
top_p_df[["top_p", "response_preview"]]

---
## 3. Max Tokens Impact

Max tokens is a hard limit on output length. Too short truncates mid-sentence; too long wastes tokens.

In [ ]:
MAX_TOKENS_VALUES = [20, 50, 100, 200]
PROMPT_MAX = "Write a Python function to calculate fibonacci numbers."

max_tokens_results = []
for max_t in MAX_TOKENS_VALUES:
    try:
        response = call_llm(
            PROMPT_MAX,
            provider=PROVIDER,
            model=MODEL,
            temperature=0.3,
            max_tokens=max_t,
        )
        max_tokens_results.append({"max_tokens": max_t, "response": response})
        print(f"\n=== Max Tokens {max_t} ===")
        print(response[:400])
    except Exception as e:
        print(f"Error at max_tokens {max_t}: {e}")
        max_tokens_results.append({"max_tokens": max_t, "response": f"ERROR: {e}"})

print(f"\nCompleted {len(max_tokens_results)} max_tokens tests.")

In [ ]:
max_df = pd.DataFrame(max_tokens_results)
max_df["response_preview"] = max_df["response"].str[:150] + "..."
max_df[["max_tokens", "response_preview"]]

---
## 4. Combined Comparison Table

Side-by-side view of all parameter experiments.

In [ ]:
comparison = []
for r in temp_results:
    comparison.append({"parameter": "temperature", "value": r["temperature"], "response": r["response"][:150]})
for r in top_p_results:
    comparison.append({"parameter": "top_p", "value": r["top_p"], "response": r["response"][:150]})
for r in max_tokens_results:
    comparison.append({"parameter": "max_tokens", "value": r["max_tokens"], "response": r["response"][:150]})

comparison_df = pd.DataFrame(comparison)
comparison_df

---
## 5. Parameter Recommendation Guide

Based on the experiments above, here is a decision guide for production use:

In [ ]:
guide = pd.DataFrame({
    "task_type": [
        "Code generation",
        "Factual Q&A",
        "Classification",
        "Content writing",
        "Brainstorming",
        "Creative writing",
    ],
    "temperature": ["0.0", "0.0-0.3", "0.0", "0.5-0.7", "0.7-1.0", "1.0-1.5"],
    "top_p": ["0.9", "0.9", "0.9", "0.9", "0.95", "1.0"],
    "max_tokens": ["4000+", "200-500", "100-200", "500-1000", "500-1000", "1000-2000"],
})
guide

---
## Summary

Key takeaways from this notebook:

1. **Temperature** is the primary knob for controlling randomness. Use 0.0 for deterministic tasks, 0.7 for general use, 1.0+ for creative tasks.
2. **Top-p** is a secondary knob. Adjust EITHER temperature OR top-p, not both.
3. **Max tokens** must be set generously enough to avoid truncation, but not so high that you waste tokens.
4. **Provider ranges differ**: OpenAI allows 0-2.0 temperature, Anthropic allows 0-1.0, reasoning models are locked at 1.0.
5. **Temperature=0 is NOT fully deterministic** across API calls due to floating-point non-determinism.

Next: [Module 2 - Essential Prompting Strategies](../02-essential-strategies/README.md)